In [1]:
!pip install optuna -q

import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)  # silenzia i log intermedi

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import kagglehub
path = kagglehub.competition_download('titanic')
print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/titanic


In [2]:
df_train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
df_test  = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

# Preprocessing base
df_train.Cabin = df_train.Cabin.fillna('U').astype(str).str[0]
df_test.Cabin  = df_test.Cabin.fillna('U').astype(str).str[0]
df_train.Embarked = df_train.Embarked.fillna('S')
df_test.Embarked  = df_test.Embarked.fillna('S')

# Age imputation per gruppo
age_medians = df_train.groupby(['Pclass', 'Sex'])['Age'].median()
global_med  = df_train['Age'].median()
def fill_age(row):
    if pd.isnull(row['Age']):
        return age_medians.get((row['Pclass'], row['Sex']), global_med)
    return row['Age']
df_train['Age'] = df_train.apply(fill_age, axis=1)
df_test['Age']  = df_test.apply(fill_age, axis=1)

# Feature engineering (identico a titanic_final — solo WomanOrChild come novità)
for df in [df_train, df_test]:
    df['FamilySize']   = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']      = (df['FamilySize'] == 1).astype(int)
    df['AgeBand']      = pd.cut(df['Age'], bins=5, labels=False)
    df['Fare_Log']     = np.log1p(df['Fare'])
    df['WomanOrChild'] = ((df['Sex'] == 'female') | (df['Age'] < 15)).astype(int)

# Drop + split
drop_cols = ['PassengerId', 'Ticket', 'Name', 'Fare']
df_train = df_train.drop(columns=drop_cols)
df_test  = df_test.drop(columns=drop_cols)

x = df_train.drop(['Survived'], axis=1)
y = df_train['Survived']

# Encoding + Imputation
object_cols = list(x.select_dtypes('object').columns)
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
x[object_cols] = ordinal_encoder.fit_transform(x[object_cols])
imputer = SimpleImputer(strategy='median')
x_transformed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

print("Dati pronti. Shape:", x_transformed.shape)
print("Feature:", list(x_transformed.columns))

Dati pronti. Shape: (891, 12)
Feature: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Cabin', 'Embarked', 'FamilySize', 'IsAlone', 'AgeBand', 'Fare_Log', 'WomanOrChild']


In [3]:
# ── Optuna: tuning automatico degli iperparametri ─────────────────────────────
#
# Come funziona:
#   - Ogni "trial" propone una combinazione di iperparametri
#   - La valuta con cross-validation a 5 fold sul training set
#   - Usa l'algoritmo TPE (Tree-structured Parzen Estimator) per concentrarsi
#     sulle zone dello spazio dei parametri che hanno dato i risultati migliori
#   - Dopo N trial restituisce la combinazione ottimale trovata
#
# Vantaggi rispetto a GridSearchCV:
#   - Non esplora tutte le combinazioni (esponenzialmente tante), ma impara
#     quali parametri contano di più e dove cercare
#   - Molto più efficiente: 100 trial Optuna ≈ migliaia di combinazioni grid

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    # ── Spazio di ricerca XGBoost ──
    xgb_params = dict(
        n_estimators      = trial.suggest_int("xgb_n_estimators", 100, 600),
        learning_rate     = trial.suggest_float("xgb_lr", 0.01, 0.2, log=True),
        max_depth         = trial.suggest_int("xgb_max_depth", 3, 7),
        subsample         = trial.suggest_float("xgb_subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("xgb_colsample", 0.6, 1.0),
        min_child_weight  = trial.suggest_int("xgb_min_child_weight", 1, 10),
        gamma             = trial.suggest_float("xgb_gamma", 0.0, 1.0),
        random_state=0, eval_metric='logloss', verbosity=0
    )

    # ── Spazio di ricerca LightGBM ──
    lgbm_params = dict(
        n_estimators      = trial.suggest_int("lgbm_n_estimators", 100, 600),
        learning_rate     = trial.suggest_float("lgbm_lr", 0.01, 0.2, log=True),
        max_depth         = trial.suggest_int("lgbm_max_depth", 3, 7),
        subsample         = trial.suggest_float("lgbm_subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("lgbm_colsample", 0.6, 1.0),
        min_child_samples = trial.suggest_int("lgbm_min_child_samples", 5, 50),
        random_state=0, verbose=-1
    )

    # ── Spazio di ricerca RandomForest ──
    rf_params = dict(
        n_estimators  = trial.suggest_int("rf_n_estimators", 100, 600),
        max_depth     = trial.suggest_int("rf_max_depth", 3, 10),
        min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10),
        max_features  = trial.suggest_categorical("rf_max_features", ["sqrt", "log2"]),
        random_state=0
    )

    model = VotingClassifier(
        estimators=[
            ('xgb',  XGBClassifier(**xgb_params)),
            ('lgbm', LGBMClassifier(**lgbm_params)),
            ('rf',   RandomForestClassifier(**rf_params))
        ],
        voting='soft'
    )

    scores = cross_val_score(model, x_transformed, y, cv=cv, scoring='accuracy', n_jobs=-1)
    return scores.mean()

# Lancia la ricerca: 80 trial (~10-15 minuti su Kaggle)
# Aumenta n_trials per una ricerca più approfondita (ma più lenta)
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=True)

print(f"\nMigliore CV trovata: {study.best_value:.4f}")
print(f"\nMigliori parametri:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/80 [00:00<?, ?it/s]


Migliore CV trovata: 0.8518

Migliori parametri:
  xgb_n_estimators: 560
  xgb_lr: 0.1053618381277937
  xgb_max_depth: 7
  xgb_subsample: 0.937657037101308
  xgb_colsample: 0.8518139576373508
  xgb_min_child_weight: 6
  xgb_gamma: 0.40245723937441014
  lgbm_n_estimators: 191
  lgbm_lr: 0.0977352754376232
  lgbm_max_depth: 7
  lgbm_subsample: 0.6269902336455775
  lgbm_colsample: 0.9338891080665005
  lgbm_min_child_samples: 45
  rf_n_estimators: 268
  rf_max_depth: 5
  rf_min_samples_leaf: 3
  rf_max_features: log2


In [4]:
# Ricostruisce il modello con i parametri ottimali trovati da Optuna
p = study.best_params

best_xgb = XGBClassifier(
    n_estimators     = p['xgb_n_estimators'],
    learning_rate    = p['xgb_lr'],
    max_depth        = p['xgb_max_depth'],
    subsample        = p['xgb_subsample'],
    colsample_bytree = p['xgb_colsample'],
    min_child_weight = p['xgb_min_child_weight'],
    gamma            = p['xgb_gamma'],
    random_state=0, eval_metric='logloss', verbosity=0
)
best_lgbm = LGBMClassifier(
    n_estimators     = p['lgbm_n_estimators'],
    learning_rate    = p['lgbm_lr'],
    max_depth        = p['lgbm_max_depth'],
    subsample        = p['lgbm_subsample'],
    colsample_bytree = p['lgbm_colsample'],
    min_child_samples= p['lgbm_min_child_samples'],
    random_state=0, verbose=-1
)
best_rf = RandomForestClassifier(
    n_estimators     = p['rf_n_estimators'],
    max_depth        = p['rf_max_depth'],
    min_samples_leaf = p['rf_min_samples_leaf'],
    max_features     = p['rf_max_features'],
    random_state=0
)

best_model = VotingClassifier(
    estimators=[('xgb', best_xgb), ('lgbm', best_lgbm), ('rf', best_rf)],
    voting='soft'
)

# Verifica sul validation set
x_train, x_valid, y_train, y_valid = train_test_split(
    x_transformed, y, test_size=0.2, random_state=0, stratify=y)
best_model.fit(x_train, y_train)
print(f"Validation accuracy: {accuracy_score(y_valid, best_model.predict(x_valid)):.4f}")

Validation accuracy: 0.8212


In [5]:
# Training finale su tutti i dati
best_model.fit(x_transformed, y)

# Preprocessing test set
df_test[object_cols] = ordinal_encoder.transform(df_test[object_cols])
df_test_transformed  = pd.DataFrame(imputer.transform(df_test), columns=df_test.columns)

predictions = best_model.predict(df_test_transformed)

output = pd.DataFrame({
    'PassengerId': pd.read_csv('/kaggle/input/competitions/titanic/test.csv')['PassengerId'],
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Submission salvata! Shape:", output.shape)
print(output['Survived'].value_counts())

Submission salvata! Shape: (418, 2)
Survived
0    276
1    142
Name: count, dtype: int64
